# Imports


In [ ]:
import numpy as np
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from enum import Enum
from google.colab import files
uploaded = files.upload()
from gmm import GMM,CovarianceType
from Autoencoder import Autoencoder
from kmeans import NumpyKMeans

## data setting


In [ ]:

np.random.seed(42)

# Example dataset (replace with your actual dataset)
X = np.random.randn(1000, 784)

bottleneck_sizes = [2, 5, 10, 15, 20]

# Store results
ae_results = {}
pca_results = {}

## Experiment 5: K-Means after Autoencoder

In [ ]:
for bsize in bottleneck_sizes:
    print(f"\nTraining Autoencoder with bottleneck size {bsize}...")
    ae = Autoencoder(input_dim=784, hidden_dims=[256,128,64], bottleneck_dim=bsize, activation='relu')
    ae.train(X, epochs=50, batch_size=64)

    # Encode using autoencoder
    Z_ae = ae.encode(X)

    # K-Means on AE features
    kmeans_ae = NumpyKMeans(n_clusters=10, init='k-means++', max_iter=300, tol=1e-4, random_state=42)
    kmeans_ae.fit(Z_ae)
    kmeans_ae_labels = kmeans_ae.labels_

    # Reconstruction loss
    X_hat = ae.decode(Z_ae)
    recon_loss = np.mean((X - X_hat)**2)

    # K-Means clustering score (silhouette)
    kmeans_score = silhouette_score(Z_ae, kmeans_ae_labels)

    ae_results[bsize] = {
        "recon_loss": recon_loss,
        "kmeans_silhouette": kmeans_score
    }

    # PCA for comparison
    pca = PCA(n_components=bsize)
    Z_pca = pca.fit_transform(X)

    kmeans_pca = NumpyKMeans(n_clusters=10, init='k-means++', max_iter=300, tol=1e-4, random_state=42)
    kmeans_pca.fit(Z_pca)

    X_hat_pca = pca.inverse_transform(Z_pca)
    recon_loss_pca = np.mean((X - X_hat_pca)**2)
    kmeans_score_pca = silhouette_score(Z_pca, kmeans_pca.labels_)

    pca_results[bsize] = {
        "recon_loss": recon_loss_pca,
        "kmeans_silhouette": kmeans_score_pca
    }

## Experiment 6: GMM after Autoencoder

In [ ]:
gmm_results = {}
gmm_pca_results = {}

for bsize in bottleneck_sizes:
    # Autoencoder features
    Z_ae = ae.encode(X)

    gmm_ae = GMM(n_components=10, cov_type=CovarianceType.FULL, tol=1e-4, max_iter=100)
    gmm_ae.fit(Z_ae)
    gmm_ae_labels = gmm_ae.predict(Z_ae)

    gmm_score = silhouette_score(Z_ae, gmm_ae_labels)
    gmm_results[bsize] = gmm_score

    # PCA features
    pca = PCA(n_components=bsize)
    Z_pca = pca.fit_transform(X)

    gmm_pca = GMM(n_components=10, cov_type=CovarianceType.FULL, tol=1e-4, max_iter=100)
    gmm_pca.fit(Z_pca)
    gmm_pca_labels = gmm_pca.predict(Z_pca)

    gmm_score_pca = silhouette_score(Z_pca, gmm_pca_labels)
    gmm_pca_results[bsize] = gmm_score_pca


## Autoencoder vs PCA: K-Means Results

In [ ]:
for bsize in bottleneck_sizes:
    print(f"Bottleneck {bsize}: AE Loss={ae_results[bsize]['recon_loss']:.4f}, AE Silhouette={ae_results[bsize]['kmeans_silhouette']:.4f}, "
          f"PCA Loss={pca_results[bsize]['recon_loss']:.4f}, PCA Silhouette={pca_results[bsize]['kmeans_silhouette']:.4f}")

## Autoencoder vs PCA: GMM Results



In [ ]:
for bsize in bottleneck_sizes:
    print(f"Bottleneck {bsize}: AE Silhouette={gmm_results[bsize]:.4f}, PCA Silhouette={gmm_pca_results[bsize]:.4f}")